# 2. Seaborn Evaluation Plots

Generate standard evaluation plots using the **seaborn** (matplotlib) backend:
- Global bar charts (column_mapping_accuracy, avg_value_accuracy, avg_value_f1)
- Per-column performance heatmap
- Confusion matrices per run per column
- Cross-model comparison heatmaps
- Boxplots by model family, local/cloud, cost tier

In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import Image, display

os.chdir("/hpc/compgen/projects/llm_GEO_project/harmonia_metadata_agent/analysis/dstoker/harmonia")
PYTHON = ".venv/bin/python"
RESULTS_GLOB = "results/*/metrics.json"

## Step 1: Calculate metrics for all completed runs

In [ ]:
# Calculate metrics for each result directory that has a trace.json but no metrics.json
results_dir = Path("results")
for run_dir in sorted(results_dir.iterdir()):
    if not run_dir.is_dir() or run_dir.name in ("old", "older"):
        continue
    metrics_f = run_dir / "metrics.json"
    trace_f = run_dir / "trace.json"
    if trace_f.exists() and not metrics_f.exists():
        print(f"Calculating metrics for {run_dir.name}...")
        r = subprocess.run(
            [PYTHON, "calculate_metrics.py", "--results-dir", str(run_dir), "--verbose"],
            capture_output=True, text=True,
        )
        if r.returncode == 0:
            print("  OK")
        else:
            print(f"  FAILED (rc={r.returncode})")
            print(r.stderr[-500:] if r.stderr else r.stdout[-500:])
    elif metrics_f.exists():
        print(f"  {run_dir.name}: metrics.json already exists")

## Step 2: Generate standard seaborn plots

In [ ]:
from datetime import datetime

out_dir = f"analysis/plots_seaborn_{datetime.now().strftime('%Y%m%d_%H%M')}"

cmd = [
    PYTHON, "src/evaluation/make_standard_evaluation_plots.py",
    "--metrics-glob", RESULTS_GLOB,
    "--out-dir", out_dir,
    "--backend", "seaborn",
    "--figure-format", "png",
    "--backfill-row-values",
    "--verbose",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
else:
    print(f"\nPlots saved to: {out_dir}")

## Step 3: Display generated plots

In [ ]:
# Find and display all PNG plots
plots_dir = Path(out_dir) / "plots"
if plots_dir.exists():
    png_files = sorted(plots_dir.glob("*.png"))
    print(f"Found {len(png_files)} top-level plots\n")
    for png in png_files:
        print(f"--- {png.name} ---")
        display(Image(filename=str(png), width=800))
else:
    print(f"No plots directory at {plots_dir}")

## Step 4: Display confusion matrices (sample)

In [ ]:
# Show first 3 confusion matrices from each model subfolder
cm_dir = Path(out_dir) / "plots" / "confusion_matrices"
if cm_dir.exists():
    for model_dir in sorted(cm_dir.iterdir()):
        if model_dir.is_dir():
            pngs = sorted(model_dir.glob("*.png"))[:3]
            for png in pngs:
                print(f"--- {model_dir.name}/{png.name} ---")
                display(Image(filename=str(png), width=600))
else:
    print("No confusion matrices directory found")

## Step 5: Inspect saved data tables

In [ ]:
import pandas as pd

tables_dir = Path(out_dir) / "tables"
if tables_dir.exists():
    for csv_f in sorted(tables_dir.glob("*.csv")):
        df = pd.read_csv(csv_f)
        print(f"\n{'='*60}")
        print(f"{csv_f.name}: {df.shape[0]} rows x {df.shape[1]} cols")
        print(f"{'='*60}")
        display(df.head(10))
else:
    print("No tables directory found")